# Unified Modelling Pipeline

This notebook trains a **pooled (unified) model** across all tea segments using engineered features from `tea_preprocessed_v2.csv` to produce a **one-week-ahead forecast**.

## Key Features:
1. **Engineered Features Required**: Uses `tea_preprocessed_v2.csv` as the modeling base
2. **Single Toggle**: Supply-demand features can be enabled/disabled
3. **Same 4 Models as Segment-Wise**: XGBoost, LightGBM, Gradient Boosting, Random Forest
4. **5-Fold TimeSeriesSplit CV**: Chronological ordering to prevent data leakage
5. **Segment-wise Evaluation**: Evaluate unified model forecast performance on each segment separately

## Research Question:
What is the performance impact of adding supply-demand structure on top of an engineered-feature unified model for **next-week price forecasting**?

**Note**: Comparison with segment-specific modeling is handled in a separate notebook (unified_vs_segment_comparison.ipynb).

## 1. Imports and Configuration

In [1]:
"""Imports and global config for unified time-series modeling."""

import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError("Please install xgboost: pip install xgboost") from exc

try:
    from lightgbm import LGBMRegressor
except ImportError as exc:
    raise ImportError("Please install lightgbm: pip install lightgbm") from exc

SEED = 42
np.random.seed(SEED)


USE_SUPPLY_DEMAND_FEATURES = False


print(f"Configuration:")
print(f"  USE_SUPPLY_DEMAND_FEATURES: {USE_SUPPLY_DEMAND_FEATURES}")
print(f"  SEED: {SEED}")
print("Libraries loaded.")

Configuration:
  USE_SUPPLY_DEMAND_FEATURES: False
  SEED: 42
Libraries loaded.


## 2. Load Data

In [2]:
"""Load preprocessed data with robust path detection and build one-week-ahead target."""

def find_repo_root(start_path: Path | None = None) -> Path:
    """Find the repository root by searching upward for the data directory."""
    candidates = []
    if start_path is not None:
        candidates.append(start_path)
    candidates.extend([Path.cwd(), Path.cwd().parent])
    for candidate in candidates:
        for root in [candidate, *candidate.parents]:
            if (root / 'data').exists() and (root / 'src').exists():
                return root
    return Path.cwd()

repo_root = find_repo_root()
processed_dir = repo_root / 'data' / 'Processed'
reports_dir = repo_root / 'reports' / 'figures'
tables_dir = repo_root / 'reports' / 'tables'

reports_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

# Base dataset is always the engineered-feature table.
base_input_path = processed_dir / 'tea_preprocessed_v2.csv'
if not base_input_path.exists():
    raise FileNotFoundError('Required engineered dataset not found: tea_preprocessed_v2.csv')

df_raw = pd.read_csv(base_input_path)
print(f'Loaded base dataset: {base_input_path.name}')
print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns')

TARGET_BASE = 'price_mid_lkr'
TARGET = 'price_mid_lkr_t_plus_1w'
FORECAST_HORIZON = 't_plus_1_week'
SEGMENT_COL = 'category_type'

df = df_raw.copy()

# Optional supply-demand enrichment from T1.3 processed dataset.
if USE_SUPPLY_DEMAND_FEATURES:
    sd_input_path = processed_dir / 'tea_preprocessed_t13_supply_demand.csv'
    sd_cols = ['supply_pressure_index', 'demand_intensity_ratio', 'market_tightness_indicator']
    if sd_input_path.exists():
        sd_df = pd.read_csv(sd_input_path)
        merge_keys = [k for k in ['sale_id', 'category_type'] if k in df.columns and k in sd_df.columns]
        if merge_keys:
            sd_keep = merge_keys + [c for c in sd_cols if c in sd_df.columns]
            sd_unique = sd_df[sd_keep].drop_duplicates(subset=merge_keys)
            df = df.merge(sd_unique, on=merge_keys, how='left')
            print(f'Enriched with supply-demand features from: {sd_input_path.name}')
        else:
            print('Warning: Could not find compatible merge keys for supply-demand enrichment.')
            USE_SUPPLY_DEMAND_FEATURES = False
    else:
        print('Warning: Supply-demand dataset not found; continuing without supply-demand features.')
        USE_SUPPLY_DEMAND_FEATURES = False

# Basic cleaning first.
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[TARGET_BASE, SEGMENT_COL]).copy()

# Stable chronological ordering before creating t+1 target.
if 'sale_year' in df.columns:
    df['_sale_year_num'] = pd.to_numeric(df['sale_year'], errors='coerce')
else:
    df['_sale_year_num'] = np.nan
if 'sale_number' in df.columns:
    df['_sale_num'] = pd.to_numeric(df['sale_number'], errors='coerce')
else:
    df['_sale_num'] = np.nan
df['_sale_id_norm'] = pd.to_numeric(df.get('sale_id', np.nan), errors='coerce')

sort_cols = ['_sale_year_num', '_sale_num', '_sale_id_norm']
df = df.sort_values(sort_cols, kind='mergesort').reset_index(drop=True)

# One-week-ahead target: predict next auction's price within each segment stream.
df[TARGET] = df.groupby(SEGMENT_COL, sort=False)[TARGET_BASE].shift(-1)
df = df.dropna(subset=[TARGET]).copy()
df = df.drop(columns=['_sale_year_num', '_sale_num', '_sale_id_norm'])

print(f'Rows after forecast-target construction ({FORECAST_HORIZON}): {len(df):,}')
print(f'Segments: {sorted(df[SEGMENT_COL].unique())}')

Loaded base dataset: tea_preprocessed_v2.csv
Shape: 3,034 rows x 244 columns
Rows after forecast-target construction (t_plus_1_week): 2,882
Segments: ['dust', 'high_grown', 'low_grown', 'off_grade']


## 3. Feature Selection and Validation

In [3]:
"""Detect and validate engineered and optional supply-demand features."""

# Engineered features are required in this notebook.
fe_interact = [c for c in df.columns if c.startswith('interact__')]
fe_roll = [c for c in df.columns if c.startswith('roll3_')]
fe_poly = [c for c in df.columns if c.startswith('poly2__')]

if not (fe_interact and fe_roll and fe_poly):
    raise ValueError('Engineered features are required but not found in tea_preprocessed_v2.csv')
print("Engineered features detected (required):")
print(f"  interact: {len(fe_interact)}, roll3: {len(fe_roll)}, poly2: {len(fe_poly)}")

# Build structural features for all runs
grade_rank_map = {
    "bop": 4, "bopf": 4, "bop1": 4,
    "fbop": 3, "fbop1": 3, "fbopf": 3, "fbopf1": 3, "fbopf_tippy": 3,
    "op": 2, "op1": 2, "opa": 2,
    "pekoe": 1, "pek1": 1, "pekoe_fbop": 1,
}
tier_rank_map = {"select_best": 4, "best": 3, "below_best": 2, "others": 1}
prestige_rank_map = {"high": 3, "medium": 2, "low": 1}

df["grade_rank"] = df.get("grade", pd.Series("", index=df.index)).astype(str).str.lower().map(grade_rank_map).fillna(0)
df["tier_rank"] = df.get("tier", pd.Series("", index=df.index)).astype(str).str.lower().map(tier_rank_map).fillna(df.get("tier_enc", 0)).fillna(0)
df["prestige_rank"] = df.get("elevation", pd.Series("", index=df.index)).astype(str).str.lower().map(prestige_rank_map).fillna(0)

# Lagged weather variables
lag_weather_cols = [
    c for c in df.columns
    if ("lag1" in c or "lag2" in c or "lag3" in c)
    and ("precipitation" in c or "sunshine" in c or "temperature" in c)
]
print(f"Lagged weather features found: {len(lag_weather_cols)}")

# Supply-demand features (optional)
supply_demand_cols = []
if USE_SUPPLY_DEMAND_FEATURES:
    supply_demand_cols = ['supply_pressure_index', 'demand_intensity_ratio', 'market_tightness_indicator']
    supply_demand_cols = [c for c in supply_demand_cols if c in df.columns]
    if supply_demand_cols:
        print(f"Supply-demand features found: {len(supply_demand_cols)}")
    else:
        print("Warning: Supply-demand features requested but not found in dataset.")
        USE_SUPPLY_DEMAND_FEATURES = False

# Build feature list
drop_cols = {
    TARGET_BASE, TARGET, 'price_mid_lkr_log', 'price_mid_usd', 'price_lo_lkr', 'price_hi_lkr',
    'price_range_lkr', 'sale_id', 'sale_date_raw', 'has_price_target',
}

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
candidate_features = [c for c in numeric_cols if c not in drop_cols]

required_features = (lag_weather_cols + ["grade_rank", "tier_rank", "prestige_rank"] + supply_demand_cols)
required_features = [c for c in required_features if c in df.columns]

# Engineered-feature setup: keep engineered expansions in the model feature set.
feature_cols = sorted(list(set(candidate_features + required_features)))

print(f"\nTotal features for unified modeling: {len(feature_cols)}")
print("  With engineered features: True (fixed by design)")
print(f"  With supply-demand features: {USE_SUPPLY_DEMAND_FEATURES}")
print(f"  Forecast target: {TARGET} ({FORECAST_HORIZON})")

Engineered features detected (required):
  interact: 28, roll3: 19, poly2: 15
Lagged weather features found: 36

Total features for unified modeling: 231
  With engineered features: True (fixed by design)
  With supply-demand features: False
  Forecast target: price_mid_lkr_t_plus_1w (t_plus_1_week)


## 4. Data Preparation for Unified Modeling

In [4]:
"""Prepare data for pooled (unified) one-week-ahead modeling with chronological sorting."""

# Keep chronological order for TimeSeriesSplit.
sort_cols = [c for c in ["sale_year", "sale_number"] if c in df.columns]
if sort_cols:
    df = df.sort_values(sort_cols).reset_index(drop=True)
else:
    print("Warning: Could not find sale_year/sale_number for chronological sorting.")

# Extract features and one-week-ahead target
X = df[feature_cols].copy()
y = df[TARGET].copy()

# Ensure target is numeric and remove NaN
y = pd.to_numeric(y, errors='coerce')
valid_mask = y.notna()
X = X[valid_mask].reset_index(drop=True)
y = y[valid_mask].reset_index(drop=True)
df_supervised = df[valid_mask].reset_index(drop=True)

print(f"Unified dataset prepared:")
print(f"  Total rows: {len(X):,}")
print(f"  Features: {len(feature_cols)}")
print(f"  Forecast target: {TARGET} ({FORECAST_HORIZON})")
print(f"  Target shape: {y.shape}")
print(f"\nSegment distribution:")
for seg in sorted(df_supervised[SEGMENT_COL].unique()):
    seg_rows = (df_supervised[SEGMENT_COL] == seg).sum()
    print(f"  {seg}: {seg_rows:,} rows")

Unified dataset prepared:
  Total rows: 2,882
  Features: 231
  Forecast target: price_mid_lkr_t_plus_1w (t_plus_1_week)
  Target shape: (2882,)

Segment distribution:
  dust: 522 rows
  high_grown: 515 rows
  low_grown: 1,347 rows
  off_grade: 498 rows


## 5. Model Definitions (Same as Segment-Wise Pipeline)

In [5]:
"""Define the same 4 models as segment-wise modeling for direct comparison."""

models = {
    "XGBoost": XGBRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        n_jobs=-1,
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=-1,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=3,
        random_state=SEED,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
    ),
}

def compute_metrics(y_true, y_pred):
    """Compute RMSE, MAE, MAPE, and R2 on raw LKR prices."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-9))) * 100
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, mape, r2

print(f"Models defined: {list(models.keys())}")

Models defined: ['XGBoost', 'LightGBM', 'Gradient Boosting', 'Random Forest']


## 6. Unified Model Training with 5-Fold TimeSeriesSplit CV

In [6]:
"""Train unified model with 5-fold TimeSeriesSplit cross-validation for one-week-ahead target."""

print(f"Running 5-fold TimeSeriesSplit on UNIFIED (pooled) data for {FORECAST_HORIZON} forecasting...\n")

tscv = TimeSeriesSplit(n_splits=5)
all_unified_fold_results = []

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    for model_name, model in models.items():
        pipe = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", clone(model)),
        ])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        rmse, mae, mape, r2 = compute_metrics(y_test, preds)

        all_unified_fold_results.append({
            "Fold": fold_idx,
            "Model": model_name,
            "RMSE": rmse,
            "MAE": mae,
            "MAPE": mape,
            "R2": r2,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "Forecast_Horizon": FORECAST_HORIZON,
        })

unified_fold_results = pd.DataFrame(all_unified_fold_results)

# Compute fold-level summary
unified_summary = (
    unified_fold_results.groupby("Model", as_index=False)
    .agg(
        RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
        MAPE_mean=("MAPE", "mean"), MAPE_std=("MAPE", "std"),
        R2_mean=("R2", "mean"), R2_std=("R2", "std"),
    )
    .sort_values("RMSE_mean")
    .reset_index(drop=True)
)
unified_summary["Forecast_Horizon"] = FORECAST_HORIZON

print("Unified Model 5-Fold CV Summary (sorted by RMSE):")
display(unified_summary.round(4))

# Build an all-model findings table with ranks and stability indicators.
unified_all_model_findings = unified_summary.copy()
unified_all_model_findings['RMSE_rank'] = unified_all_model_findings['RMSE_mean'].rank(method='dense', ascending=True).astype(int)
unified_all_model_findings['R2_rank'] = unified_all_model_findings['R2_mean'].rank(method='dense', ascending=False).astype(int)
unified_all_model_findings['RMSE_cv_pct'] = (unified_all_model_findings['RMSE_std'] / unified_all_model_findings['RMSE_mean']) * 100.0
unified_all_model_findings['R2_cv_pct'] = (unified_all_model_findings['R2_std'].abs() / unified_all_model_findings['R2_mean'].abs().replace(0, np.nan)) * 100.0
unified_all_model_findings = unified_all_model_findings.sort_values(['RMSE_rank', 'R2_rank']).reset_index(drop=True)

print('All-Model Findings Table (not limited to best model):')
display(unified_all_model_findings.round(4))

Running 5-fold TimeSeriesSplit on UNIFIED (pooled) data for t_plus_1_week forecasting...



Unified Model 5-Fold CV Summary (sorted by RMSE):


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,Forecast_Horizon
0,XGBoost,298.9183,69.0976,178.7275,26.4287,15.2594,1.6244,0.7682,0.1032,t_plus_1_week
1,Random Forest,304.4743,80.3303,171.0197,25.0210,14.1840,1.3159,0.7563,0.1237,t_plus_1_week
2,Gradient Boosting,316.5722,55.6263,195.0237,19.9802,16.6748,1.5536,0.7443,0.0849,t_plus_1_week
3,LightGBM,321.5819,70.9785,194.0501,31.6119,16.2366,1.9135,0.7303,0.1164,t_plus_1_week


All-Model Findings Table (not limited to best model):


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,Forecast_Horizon,RMSE_rank,R2_rank,RMSE_cv_pct,R2_cv_pct
0,XGBoost,298.9183,69.0976,178.7275,26.4287,15.2594,1.6244,0.7682,0.1032,t_plus_1_week,1,1,23.1159,13.4349
1,Random Forest,304.4743,80.3303,171.0197,25.0210,14.1840,1.3159,0.7563,0.1237,t_plus_1_week,2,2,26.3833,16.3549
2,Gradient Boosting,316.5722,55.6263,195.0237,19.9802,16.6748,1.5536,0.7443,0.0849,t_plus_1_week,3,3,17.5714,11.4097
3,LightGBM,321.5819,70.9785,194.0501,31.6119,16.2366,1.9135,0.7303,0.1164,t_plus_1_week,4,4,22.0717,15.9375


## 7. Segment-Wise Evaluation of Unified Model

In [7]:
"""Evaluate unified model forecast performance by segment using out-of-fold (OOF) predictions only."""

# Select best unified model from CV summary, then generate fold-wise OOF predictions.
best_model_name = unified_summary.iloc[0]["Model"]
best_model_config = models[best_model_name]

oof_pred = np.full(len(X), np.nan, dtype=float)
tscv_oof = TimeSeriesSplit(n_splits=5)

for fold_idx, (train_idx, test_idx) in enumerate(tscv_oof.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train = y.iloc[train_idx]

    pipe_best = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", clone(best_model_config)),
    ])
    pipe_best.fit(X_train, y_train)
    oof_pred[test_idx] = pipe_best.predict(X_test)

# Attach OOF predictions to supervised dataframe.
df_supervised["unified_pred_lkr"] = oof_pred
df_supervised["unified_residual"] = df_supervised[TARGET] - df_supervised["unified_pred_lkr"]
df_supervised["unified_abs_error"] = np.abs(df_supervised["unified_residual"])

oof_mask = np.isfinite(df_supervised["unified_pred_lkr"].values)

# Evaluate by segment on OOF rows only (fair out-of-sample metrics).
unified_segment_results = []
for segment in sorted(df_supervised[SEGMENT_COL].unique()):
    seg_mask = (df_supervised[SEGMENT_COL] == segment) & oof_mask
    seg_y_true = df_supervised.loc[seg_mask, TARGET].values
    seg_y_pred = df_supervised.loc[seg_mask, "unified_pred_lkr"].values

    if len(seg_y_true) == 0:
        continue

    rmse, mae, mape, r2 = compute_metrics(seg_y_true, seg_y_pred)
    unified_segment_results.append({
        "Segment": segment,
        "Model": "Unified",
        "N_Rows": int((df_supervised[SEGMENT_COL] == segment).sum()),
        "N_Rows_OOF": int(len(seg_y_true)),
        "Eval_Scope": "OOF_TimeSeriesCV",
        "Forecast_Horizon": FORECAST_HORIZON,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2,
    })

unified_by_segment = pd.DataFrame(unified_segment_results)

print(f"\nBest Unified Model: {best_model_name}")
print(f"Forecast horizon: {FORECAST_HORIZON}")
print(f"OOF coverage: {oof_mask.sum():,}/{len(oof_mask):,} rows ({100.0 * oof_mask.mean():.2f}%)")
print(f"\nUnified OOF Performance by Segment:")
display(unified_by_segment.round(4))


Best Unified Model: XGBoost
Forecast horizon: t_plus_1_week
OOF coverage: 2,400/2,882 rows (83.28%)

Unified OOF Performance by Segment:


,Segment,Model,N_Rows,N_Rows_OOF,Eval_Scope,Forecast_Horizon,RMSE,MAE,MAPE,R2
0,dust,Unified,522,419,OOF_TimeSeriesCV,t_plus_1_week,176.2516,134.8095,15.7696,0.2938
1,high_grown,Unified,515,420,OOF_TimeSeriesCV,t_plus_1_week,233.9280,170.7016,19.1523,0.2156
2,low_grown,Unified,1347,1139,OOF_TimeSeriesCV,t_plus_1_week,395.4698,222.1303,13.3429,0.7271
3,off_grade,Unified,498,422,OOF_TimeSeriesCV,t_plus_1_week,149.8603,113.1750,16.0509,0.2529


## 8. Save Results

In [8]:
"""Save unified model results to disk."""

config_suffix = "_with_supply_demand" if USE_SUPPLY_DEMAND_FEATURES else "_no_supply_demand"

unified_fold_results.to_csv(
    tables_dir / f"unified_cv_fold_results{config_suffix}.csv",
    index=False,
)
unified_summary.to_csv(
    tables_dir / f"unified_cv_summary{config_suffix}.csv",
    index=False,
)
unified_all_model_findings.to_csv(
    tables_dir / f"unified_all_model_findings{config_suffix}.csv",
    index=False,
)
unified_by_segment.to_csv(
    tables_dir / f"unified_segment_evaluation{config_suffix}.csv",
    index=False,
)

print("Saved:")
print(f"- unified_cv_fold_results{config_suffix}.csv")
print(f"- unified_cv_summary{config_suffix}.csv")
print(f"- unified_all_model_findings{config_suffix}.csv")
print(f"- unified_segment_evaluation{config_suffix}.csv")

Saved:
- unified_cv_fold_results_no_supply_demand.csv
- unified_cv_summary_no_supply_demand.csv
- unified_all_model_findings_no_supply_demand.csv
- unified_segment_evaluation_no_supply_demand.csv


## 9. Configuration Summary and Results

In [9]:
# Save the key report-ready outputs.
tables_dir.mkdir(parents=True, exist_ok=True)

unified_summary.to_csv(
    tables_dir / f"unified_cv_summary{config_suffix}.csv",
    index=False,
)
unified_all_model_findings.to_csv(
    tables_dir / f"unified_all_model_findings{config_suffix}.csv",
    index=False,
)
unified_by_segment.to_csv(
    tables_dir / f"unified_segment_evaluation{config_suffix}.csv",
    index=False,
)

config_path = tables_dir / f"unified_pipeline_config{config_suffix}.json"
config_payload = {
    "engineered_feature_source": "tea_preprocessed_v2.csv",
    "use_engineered_features": True,
    "use_supply_demand_features": "with_supply_demand" in config_suffix,
    "config_suffix": config_suffix,
    "target_base": TARGET_BASE,
    "target_column": TARGET,
    "forecast_horizon": FORECAST_HORIZON,
}
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, indent=2)

print("Saved report-ready outputs:")
print(f"- unified_cv_summary{config_suffix}.csv")
print(f"- unified_all_model_findings{config_suffix}.csv")
print(f"- unified_segment_evaluation{config_suffix}.csv")
print(f"- unified_pipeline_config{config_suffix}.json")

Saved report-ready outputs:
- unified_cv_summary_no_supply_demand.csv
- unified_all_model_findings_no_supply_demand.csv
- unified_segment_evaluation_no_supply_demand.csv
- unified_pipeline_config_no_supply_demand.json
